In [47]:
import pandas as pd
import os
from typing import List, Dict
import pandas as pd



In [48]:
delays = pd.read_excel('audio delays.xlsx')
subjects = delays.loc[0:11]
subjects

,Subject,p1,p2,p3
0,Alireza_9044,2.04,1.99,1.92
1,Arya_3561,1.96,2.04,2.12
2,Bita_2861,2.51,2.17,2.09
3,Diba_8191,3.51,1.99,3.02
4,Erfan_3914,2.01,1.92,1.70
5,Faezeh_3703,2.29,3.10,2.07
6,Ghazal_5424,1.89,1.88,1.67
7,Ghazal_8825,2.61,2.23,1.92
8,Kamyar_564,1.92,2.28,3.09
9,Mana_2933,1.88,1.95,2.05


In [49]:
for index in subjects.index:
    subject_name = subjects.loc[index, 'Subject']
    subject_file = f'test128_{subject_name}.csv'
    subjects.loc[index, 'file_name'] = subject_file

C:\Users\MSI\AppData\Local\Temp\ipykernel_8256\3888984710.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subjects.loc[index, 'file_name'] = subject_file


In [50]:
def extract_dynamic_obstacle_trials(df):
    """
    Detects rising edge in 'is_dynamic_obstacle_present' column and returns a DataFrame
    with only the valid perception rows (where the dynamic obstacle state rises from False to True).
    The feedback_modality is taken from the row immediately after the rise.
    """
    df = df.copy()

    # Ensure boolean
    df["is_dynamic_obstacle_present"] = df["is_dynamic_obstacle_present"].astype(bool)

    # Detect rising edge: FALSE → TRUE
    df["dynamic_rise"] = (
        df["is_dynamic_obstacle_present"]
        & ~df["is_dynamic_obstacle_present"].shift(1, fill_value=False)
    )

    # Set feedback_modality from the next row
    df.loc[df["dynamic_rise"], "feedback_modality"] = (
        df["feedback_modality"].shift(-1)
    )

    # Keep only valid perception rows
    trials = df[df["dynamic_rise"]].copy()
    return trials

In [51]:
def compute_trial_delays(df : pd.DataFrame, phase_number : int) :
    if phase_number == 1 :
        start_trial = df.query('interval_number == 0 and trial_number == 0').iloc[0]
        start_cue = df.query('interval_number == 1 and trial_number == 1').iloc[0]
        start_trial_time = start_trial['timestamp']
        start_cue_time = start_cue['timestamp']
        print(f'phase1 added time: {start_cue_time -start_trial_time}')
        return start_cue_time -start_trial_time
        

    elif phase_number == 2 :
        start_trial = df.query('interval_number == 73 and trial_number == 72').iloc[0]
        start_cue = df.query('interval_number == 73 and trial_number == 73').iloc[0]
        start_trial_time = start_trial['timestamp']
        start_cue_time = start_cue['timestamp']
        print(f'phase2 added time: {start_cue_time -start_trial_time}')
        return start_cue_time -start_trial_time

    elif phase_number == 3 :
        start_trial = df.query('interval_number == 145 and trial_number == 144').iloc[0]
        start_cue = df.query('interval_number == 145 and trial_number == 145').iloc[0]
        start_trial_time = start_trial['timestamp']
        start_cue_time = start_cue['timestamp']
        print(f'phase3 added time: {start_cue_time -start_trial_time}')
        return start_cue_time -start_trial_time


In [ ]:
def create_relative_stamps(trials: pd.DataFrame , full_df: pd.DataFrame) :
    # Create a mask for reset points
    reset_mask = trials['interval_number'].isin([73, 145])
    # Get indices where resets occur
    reset_indices = trials[reset_mask].index
    # Create cumulative group based on resets
    trials['group'] = 0
    for idx in reset_indices:
        trials.loc[idx:, 'group'] += 1
    
    # Calculate relative timestamp (starts at 0 for each group)
    trials['relative_timestamp'] = trials.groupby('group')['timestamp'].transform(lambda x: x - x.iloc[0])
    
    # Add phase-specific delays
    for group_num, phase in enumerate([1, 2, 3]):
        delay = compute_trial_delays(full_df, phase)
        print(f'delay for phase {phase} is {delay}')
        group_mask = trials['group'] == group_num
        trials.loc[group_mask, 'relative_timestamp'] += delay
    
    # Clean up
    trials = trials.drop('group', axis=1)
    trials = trials.reset_index()
    return trials

In [53]:
def find_anomaly(
    df_trials: pd.DataFrame,
    perceived_values: List[Dict],
    start_idx: int,
    base_delay: float,
    tolerance: float = 0.5
):

    corrected_values = []

    # pointer over perceived_values
    pv_idx = 0

    phase_end = start_idx + 72

    print("\n================ FINDING ANOMALIES ================\n")

    for trial_idx in range(start_idx, phase_end):

        # current cue timestamp
        current_time = df_trials.loc[trial_idx, 'relative_timestamp']

        # next cue timestamp
        if trial_idx + 1 < len(df_trials):
            next_time = df_trials.loc[trial_idx + 1, 'relative_timestamp']
        else:
            next_time = current_time + 10  # arbitrary large gap for the last trial
        


        if trial_idx == 71 :
            next_time = current_time + 13
        if trial_idx == 143 :
            next_time = current_time + 13
        if next_time < current_time:
            print(f' next_time < current_time index is {trial_idx}')
            continue

        # response window
        window_start = current_time
        window_end = next_time + tolerance

        matches = []

        # collect all perceived values inside this window
        while pv_idx < len(perceived_values):

            pv = perceived_values[pv_idx]

            shifted_start = (pv['start_ms'] / 1000) + base_delay
            shifted_end = (pv['end_ms'] / 1000) + base_delay

            # speech before current window
            if shifted_end < window_start:
                # print(
                #     f"[SKIP] speech before window | "
                #     f"speech={shifted_start:.3f}-{shifted_end:.3f}"
                # )
                pv_idx += 1
                continue

            # speech after window -> stop checking
            if shifted_start > window_end:
                break

            # speech inside window
            matches.append(pv)
            pv_idx += 1

        # ====================================================
        # CASE 1: MISSING RESPONSE
        # ====================================================
        if len(matches) == 0:

            print(
                f"[MISSING] trial={trial_idx} "
                f"time={current_time:.3f} "
                f"window=({window_start:.3f}, {window_end:.3f})"
            )

            corrected_values.append({
                'degree': 9999,
                'level': 9999,
                'start_ms': 0,
                'end_ms': 0,
                'text': '9999 9999'
            })

        # ====================================================
        # CASE 2: NORMAL RESPONSE
        # ====================================================
        elif len(matches) == 1:

            corrected_values.append(matches[0])

        # ====================================================
        # CASE 3: MULTIPLE RESPONSES
        # ====================================================
        else:

            print(
                f"[DUPLICATE] trial={trial_idx} "
                f"time={current_time:.3f} "
                f"found={len(matches)} responses"
            )

            for j, m in enumerate(matches):
                shifted_start = (m['start_ms'] / 1000) + base_delay
                shifted_end = (m['end_ms'] / 1000) + base_delay

                print(
                    f"    candidate {j+1}: "
                    f"text={m['text']} "
                    f"time={shifted_start:.3f}-{shifted_end:.3f}"
                )

            # keep FIRST response only
            corrected_values.append(matches[0])

    print("\n===================================================\n")

    print(f"Original perceived_values length : {len(perceived_values)}")
    print(f"Corrected perceived_values length: {len(corrected_values)}")

    return corrected_values

In [54]:
def append_voice_stamps(df_trials :pd.DataFrame, perceived_values : Dict , base_delay : float , phase : int ) :
        # Determine the starting index based on part
    if 'degree_perceived' not in df_trials.columns:
        df_trials['degree_perceived'] = pd.NA
    if 'level_perceived' not in df_trials.columns:
        df_trials['level_perceived'] = pd.NA
    if 'voice_start' not in df_trials.columns:
        df_trials['voice_start'] = pd.NA
    if 'voice_end' not in df_trials.columns:
        df_trials['voice_end'] = pd.NA
    if phase == 1:
        start_idx = 0
    elif phase == 2:
        start_idx = df_trials[df_trials['interval_number'] == 73].index[0]
        print("part 2 ")
        print(start_idx)
    elif phase == 3:
        print("part 3")
        
        start_idx = df_trials[df_trials['interval_number'] == 145].index[0]
        print(start_idx)
    else:
        raise ValueError("Part must be 1, 2, or 3")
    
        # Parse perceived values and fill the 
    if len(perceived_values) != 72:

        print(
            f'len perceived_values is {len(perceived_values)} '
            f'performing anomaly correction'
        )
        perceived_values = find_anomaly(
                    df_trials,
                    perceived_values,
                    start_idx,
                    base_delay
                )

    for i, perceived_value in enumerate(perceived_values):
        row_idx = start_idx + i
        if row_idx >= len(df_trials):
            break
        degree_perceived, level_perceived = perceived_value['text'].split()
        df_trials.loc[row_idx, 'degree_perceived'] = int(degree_perceived)
        df_trials.loc[row_idx, 'level_perceived'] = int(level_perceived)
        # print(f'row_idx: {row_idx}')
        voice_stamp_start = perceived_values[row_idx%72]['start_ms']/1000
        voice_stamp_end = perceived_values[row_idx%72]['end_ms']/1000
        df_trials.loc[row_idx,'voice_start'] = voice_stamp_start + base_delay
        df_trials.loc[row_idx,'voice_end'] = voice_stamp_end + base_delay
    

        # Reorder columns to place perceived next to original
    cols = df_trials.columns.tolist()
    
    # Remove the perceived columns from their current position
    cols.remove('degree_perceived')
    cols.remove('level_perceived')
    
    # Find positions of degree and level
    degree_idx = cols.index('degree')
    level_idx = cols.index('level')
    
    # Insert perceived columns after their originals
    cols.insert(degree_idx + 1, 'degree_perceived')
    cols.insert(level_idx + 2, 'level_perceived')  # +2 because we already inserted degree_perceived
    
    df_trials = df_trials[cols]
    
    return df_trials


In [55]:
from voice_reader import load_and_process,merge_tokens_to_text , levels_dict , degrees_dict , extract_degree_levels

dataset_full_directory = '../s1-j-c1/'
for index,subject in subjects.iterrows():
    subject_name = subject['Subject']
    
    if subject_name != 'Alireza_9044':
        continue
    print(subject_name)
    print(f'Processing Subject ....  {subject_name}')
    subject_full_file = f'received_data_{subject_name}.csv'
    dataset_path = os.path.join(dataset_full_directory,subject['Subject'],subject_full_file)
    df_subject_full = pd.read_csv(dataset_path)
    
    df_trials = extract_dynamic_obstacle_trials(df_subject_full)
    df_trials = create_relative_stamps(df_trials,df_subject_full)
    for phase in range(1,4):
        voice_delay = subject[f'p{phase}']
        print(f"processing phase: {phase} base recording delay: {voice_delay}")
        text_arr , voice = load_and_process(f'{subject_name}/audio_{subject_name}_{phase}_full.json')
        full_text_arr = merge_tokens_to_text(text_arr , voice['tokens'])
        precived_values = extract_degree_levels(full_text_arr,degrees_dict,levels_dict)
        df_trials = append_voice_stamps(df_trials , precived_values , voice_delay , phase)
        df_trials.to_csv(f'./{subject_name}_cleaned.csv')
    # break
    

Alireza_9044
Processing Subject ....  Alireza_9044
phase1 added time: 7.77
phase2 added time: 0.43000000000006366
phase3 added time: 3.6299999999996544
processing phase: 1 base recording delay: 2.04
len result: 72
processing phase: 2 base recording delay: 1.99
len result: 72
part 2 
72
processing phase: 3 base recording delay: 1.92
len result: 71
part 3
144
len perceived_values is 71 performing anomaly correction

================ FINDING ANOMALIES ================

[MISSING] trial=181 time=374.050 window=(374.050, 381.560)


Original perceived_values length : 71
Corrected perceived_values length: 72
